In [1]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

import spacy
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

from tqdm.notebook import tqdm as tqdm_notebook
from tqdm import tqdm

from concurrent.futures import ProcessPoolExecutor,as_completed
import multiprocessing

import pathlib, os

import pandas as pd
import numpy as np

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Data Preparation

In [2]:
import re


def data_preparation(dataset:str):
       
       # Download dataset and unzip the dataset
       url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
       out_dir = os.path.join(pathlib.Path(os.path.abspath('')), "datasets")
       data_path = util.download_and_unzip(url, out_dir)
       
       # train/test split
       d_train,q_train,qr_train=GenericDataLoader(data_path).load(split="train")
       _,q_test,qr_test=GenericDataLoader(data_path).load(split="test")
       
       # to pandas dataframe
       train={"documents":pd.DataFrame.from_dict(d_train,orient="index").drop(["title"], axis=1),
              "queries":pd.DataFrame.from_dict(q_train,orient="index",columns=["text"]),
              "real":qr_train}
       
       test={"queries":pd.DataFrame.from_dict(q_test,orient="index",columns=["text"]),
              "real":qr_test}
              
              
       def data_tokenizer(train:dict[pd.DataFrame],test:dict[pd.DataFrame]):
              nlp = spacy.load("en_core_web_sm", disable=["tok2vec", "parser", "attribute_ruler", "ner"])
              
              def tokenizer(text):
                     result=[]
                     for token in nlp(text):
                            if not token.is_stop and not token.is_punct:
                                   
                                   text=re.sub("[^A-Za-z0-9]+","",token.lemma_)
                                   
                                   if text != "":
                                          result.append(text)
                     return result
              
              def cleaning(tokenization):
                     return " ".join(tokenization)
              
              tqdm_notebook.pandas(desc="Document tokenization")
              train["documents"]["tokenized_text"]=train["documents"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Document cleaning")
              train["documents"]["text"]=train["documents"]["tokenized_text"].progress_apply(cleaning)
              
              tqdm_notebook.pandas(desc="Training queries tokenization")
              train["queries"]["tokenized_text"]=train["queries"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Training queries cleaning")
              train["queries"]["text"]=train["queries"]["tokenized_text"].progress_apply(cleaning)
              
              tqdm_notebook.pandas(desc="Test queries tokenization")
              test["queries"]["tokenized_text"]=test["queries"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Test queries cleaning")
              test["queries"]["text"]=test["queries"]["tokenized_text"].progress_apply(cleaning)
       
       #print(train["documents"].equals(test["documents"]))
       #print(train["queries"].equals(test["queries"]))
       
       #data tokenization using spacy
       data_tokenizer(train,test)
       
       return train,test

train,test=data_preparation("fiqa")

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Document tokenization:   0%|          | 0/57638 [00:00<?, ?it/s]

/home/michele/.local/lib/python3.10/site-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Document cleaning:   0%|          | 0/57638 [00:00<?, ?it/s]

...

/home/michele/.local/lib/python3.10/site-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Training queries cleaning:   0%|          | 0/5500 [00:00<?, ?it/s]

Test queries tokenization:   0%|          | 0/648 [00:00<?, ?it/s]

/home/michele/.local/lib/python3.10/site-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Test queries cleaning:   0%|          | 0/648 [00:00<?, ?it/s]

Sparse Vector Score

In [7]:
bar_queue = multiprocessing.Queue()

def BM25(bm25,start,stop):
    
    scores_per_query={}

    for index,row in train["queries"].iloc[start:stop].iterrows():
        
        scores=bm25.get_scores(row["tokenized_text"])
        
        scores_per_query[index]=scores
        
        #print(np.count_nonzero(scores))
        
        bar_queue.put_nowait(1)

    return scores_per_query

def BM25_Multiprocessing(n_jobs=8):
    
    scores_per_query_total={}
    
    def bar_handler(q):
        bar=tqdm(desc="Total", total=len(train["queries"]),bar_format="{l_bar}{bar}|{n_fmt}/{total_fmt} [{elapsed}]")
        
        while True:
            how_much=q.get()
            bar.update(how_much)
            bar.refresh()

    bar_process = multiprocessing.Process(target=bar_handler, args=(bar_queue, ), daemon=True)
    bar_process.start()

    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        
        bm25 = BM25Okapi(train["documents"]["tokenized_text"].tolist())
        
        length=int(len(train["queries"])/n_jobs)
        futures=[]
            
        for start in range(0,len(train["queries"]),length):
            
            futures.append( executor.submit(BM25,bm25,start,start+length) )
        
        for future in as_completed(futures):
            result=future.result()
            scores_per_query_total.update(result)

    bar_process.terminate()
    
    return np.array([scores_per_query_total[i] for i in scores_per_query_total.keys()])

In [8]:
scores_per_query_sparse=BM25_Multiprocessing(8)

del bar_queue

Total: 100%|██████████|5500/5500 [02:46]

In [9]:
def performance( scores_per_query, k ):
    
    found=0

    for idx,res in enumerate( scores_per_query ):
        
        indexes = np.argpartition(res, -k)[-k:]  # Indices not sorted
        
        predicted_most_relevant=train["documents"].iloc[indexes[np.argsort(res[indexes])][::-1]].index.values
        
        #print(f'Predicted - Best document for query {idx} is the {train["documents"].iloc[winner].name}')
        #print(f'Ground Truth - Best document for query {idx} is the {train["real"][str(idx)]}\n')
        
        true_relevant=list(train["real"][ train["queries"].index.values[idx] ].keys())
        
        #print(true_relevant,predicted_most_relevant)
        
        if len( set(predicted_most_relevant) & set(true_relevant) )>0:
            found+=1
        
    return found/5500

In [18]:
performance(scores_per_query_sparse,k=5500)

0.27545454545454545

Dense Vector Score

In [11]:
def dense_score(device:str="cuda"):
    
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

    documents_embeddings = model.encode(train["documents"]["text"])

    queries_embeddings = model.encode(train["queries"]["text"])

    return np.dot(queries_embeddings,documents_embeddings.transpose())

In [12]:
scores_per_query_dense=dense_score()

In [19]:
performance(scores_per_query_dense,k=5500)

0.9776363636363636

Merging

In [10]:
scores_per_query_sparse

array([[0.        , 2.5216984 , 4.32309685, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 3.70883113, ..., 0.        , 2.54375752,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 4.51517946, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 1.166097  , 0.        ,
        0.        ]])

In [11]:
#scores_per_query_dense[scores_per_query_dense<0]=0
scores_per_query_dense

array([[ 0.10726716, -0.02233031,  0.11277482, ...,  0.06628048,
        -0.02032895,  0.16648611],
       [ 0.0414651 ,  0.01074784,  0.17807701, ..., -0.00112139,
        -0.00744208,  0.08031341],
       [ 0.1439148 ,  0.01809647, -0.02375477, ..., -0.03516331,
         0.03577087,  0.11387179],
       ...,
       [ 0.01888751,  0.09061822,  0.15238571, ...,  0.06977446,
         0.06311376,  0.07325356],
       [ 0.02687474, -0.02229198,  0.05686567, ...,  0.03233647,
        -0.08190214,  0.17199439],
       [ 0.00277423, -0.00707781, -0.04452303, ...,  0.09891197,
        -0.05061857,  0.00409093]], dtype=float32)

In [12]:

stop

import gc
gc.collect()

NameError: name 'stop' is not defined

In [ ]:
total=scores_per_query_dense+scores_per_query_sparse

In [ ]:
total

array([[ 1.07267156e-01, -2.23303139e-02,  1.12774819e-01, ...,
         6.62804767e-02, -2.03289539e-02,  1.66486114e-01],
       [ 4.14650999e-02,  1.07478406e-02,  1.78077012e-01, ...,
        -1.12139434e-03, -7.44207576e-03,  8.03134069e-02],
       [ 1.43914804e-01,  1.80964693e-02, -2.37547681e-02, ...,
        -3.51633132e-02,  3.57708670e-02,  1.13871790e-01],
       ...,
       [ 1.88875087e-02,  9.06182155e-02,  1.52385712e-01, ...,
         6.97744638e-02,  1.45369716e+01,  7.32535645e-02],
       [ 2.68747360e-02, -2.22919807e-02,  6.80975862e+00, ...,
         3.23364735e-02, -8.19021389e-02,  3.12788574e+00],
       [ 2.77423114e-03, -7.07780570e-03, -4.45230268e-02, ...,
         9.89119709e-02, -5.06185740e-02,  2.95998228e+00]])

In [ ]:
performance(total)

0.07890909090909091